# 02 — Keras MLP
Hiperparametre deneyleri: `build_mlp(...)` argümanlarını değiştirip son hücreyi tekrar çalıştırın.

In [ ]:
# --- Kurulum: Colab'da repoyu klonlar/günceller, yerelde repo köküne geçer ---
import os, sys, pathlib
REPO_URL = "https://github.com/aysenursimsk/force2020-litoloji.git"   # ← kendi repo adresiniz

if "google.colab" in sys.modules:
    if not pathlib.Path("/content/force2020-litoloji").exists():
        !git clone -q {REPO_URL} /content/force2020-litoloji
    %cd /content/force2020-litoloji
    !git pull -q

root = pathlib.Path.cwd()
if not (root / "src").exists():
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root))
try:
    %load_ext autoreload
    %autoreload 2
except Exception:
    print("autoreload kullanılamıyor (sorun değil): src değişirse oturumu yeniden başlatın.")

In [ ]:
KIM = ""          # ← adınız: deney kaydında görünür
N_WELLS = None    # hızlı deneme için ör. 20 (sadece ilk 20 eğitim kuyusu); final için None

In [ ]:
from src import veri, modeller as M, egitim as E, degerlendir as D

d = veri.hazirla(n_wells=N_WELLS)
d.ozet()
cw = E.sinif_agirliklari(d.y_tr)

In [ ]:
ayar = dict(n_layers=3, units=128, act="relu", dropout=0.3, l2=1e-4, optimizer="adam", lr=1e-3)
model = M.build_mlp(d.X_tr.shape[1], **ayar)
model.summary()
hist = E.egit(model, d.X_tr, d.y_tr, d.X_va, d.y_va, epochs=30, class_weight=cw)
D.ogrenme_egrisi(hist, "MLP")

In [ ]:
p = model.predict(d.X_te, batch_size=4096, verbose=0).argmax(1)
s = D.skorlar(d.y_te, p); print(s)
D.rapor(d.y_te, p)
D.confusion(d.y_te, p, "MLP")
D.kaydet("mlp_temel", "MLP", ayar | {"class_weight": True}, s, kim=KIM, epochs=len(hist.history["loss"]))

## Deney şablonu
Aynı veriyle birden fazla ayarı arka arkaya koşturur ve hepsini `sonuclar/deneyler.csv`'ye yazar.

In [ ]:
deneyler = {
    "katman_1": dict(n_layers=1),
    "katman_5": dict(n_layers=5),
    "lr_1e-2":  dict(lr=1e-2),
    "lr_1e-4":  dict(lr=1e-4),
    "duzenlilestirmesiz": dict(dropout=0.0, l2=0.0),   # overfitting'i görmek için
}
temel = dict(n_layers=3, units=128, act="relu", dropout=0.3, l2=1e-4, optimizer="adam", lr=1e-3)
for ad, degisiklik in deneyler.items():
    a = temel | degisiklik
    m = M.build_mlp(d.X_tr.shape[1], **a)
    h = E.egit(m, d.X_tr, d.y_tr, d.X_va, d.y_va, epochs=30, class_weight=cw, verbose=0)
    p = m.predict(d.X_te, batch_size=4096, verbose=0).argmax(1)
    D.kaydet(ad, "MLP", a, D.skorlar(d.y_te, p), kim=KIM, epochs=len(h.history["loss"]))